# Baseline Notebook: Customer Churn

**Author:** Aathika Begam A
**Date:** 14 November 2026

This notebook goes with my problem-framing memo and responsible data card. It does four things:

1. checks the quality of the data
2. tests a simple non-ML rule as a baseline
3. trains one small model for comparison, labelled as a **demonstration only**
4. compares the two using the cost of mistakes

**Important:** the dataset has only 12 customers. Everything below should be read with that in mind. A perfect score on 12 rows does not tell us how a method would do on new customers.

## 1. Load the data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Works whether the notebook sits next to the data or in a notebook/ folder
candidates = [
    Path("customer-churn-training.csv"),
    Path("data/customer-churn-training.csv"),
    Path("../data/customer-churn-training.csv"),
]
path = next(p for p in candidates if p.exists())

df = pd.read_csv(path)
print("Loaded from:", path)
print("Shape:", df.shape)
df

Loaded from: ../data/customer-churn-training.csv
Shape: (12, 7)


,customer_id,tenure_months,support_tickets,monthly_spend_inr,last_login_days,plan_type,churned
0,C001,2,4,499,21,Basic,1
1,C002,18,1,1299,2,Pro,0
2,C003,6,3,799,14,Basic,1
3,C004,30,0,1499,1,Pro,0
4,C005,11,2,999,5,Standard,0
5,C006,1,5,499,30,Basic,1
6,C007,24,1,999,3,Standard,0
7,C008,8,4,799,17,Basic,1
8,C009,15,2,1299,4,Pro,0
9,C010,4,3,499,12,Basic,1


## 2. Data quality checks

In [2]:
print("Missing values per column:")
print(df.isna().sum().to_string())

print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df.customer_id.duplicated().sum())

print("\nSummary of the numeric columns:")
df.describe().loc[["min", "50%", "max"]].round(1)

Missing values per column:
customer_id          0
tenure_months        0
support_tickets      0
monthly_spend_inr    0
last_login_days      0
plan_type            0
churned              0

Duplicate rows: 0
Duplicate customer IDs: 0

Summary of the numeric columns:


,tenure_months,support_tickets,monthly_spend_inr,last_login_days,churned
min,1.0,0.0,499.0,1.0,0.0
50%,9.5,2.0,899.0,7.0,0.0
max,30.0,5.0,1499.0,30.0,1.0


In [3]:
counts = df.churned.value_counts().sort_index()
print("Class balance")
print(f"  Stayed  (0): {counts[0]}  ({counts[0] / len(df):.1%})")
print(f"  Churned (1): {counts[1]}  ({counts[1] / len(df):.1%})")

Class balance
  Stayed  (0): 7  (58.3%)
  Churned (1): 5  (41.7%)


**What this shows:** no missing values and no duplicates. 5 of 12 customers churned (41.7%), which is much higher than real churn usually is.

## 3. How well do the features separate churners from loyal customers?

In [4]:
cols = ["tenure_months", "support_tickets", "monthly_spend_inr", "last_login_days"]
print("Range of each feature for customers who stayed (0) vs churned (1):\n")
summary = df.groupby("churned")[cols].agg(["min", "max"])
summary

Range of each feature for customers who stayed (0) vs churned (1):



tenure_months     support_tickets     monthly_spend_inr        \
                  min max             min max               min   max   
churned                                                                 
0                   7  30               0   2               799  1499   
1                   1   8               3   5               499   799   

        last_login_days      
                    min max  
churned                      
0                     1   9  
1                    12  30

In [5]:
print("Churn by plan type:\n")
print(pd.crosstab(df.plan_type, df.churned, rownames=["plan_type"], colnames=["churned"]))

print("\nMonthly spend range per plan:\n")
df.groupby("plan_type").monthly_spend_inr.agg(["min", "max"])

Churn by plan type:

churned    0  1
plan_type      
Basic      0  5
Pro        3  0
Standard   4  0

Monthly spend range per plan:



,min,max
plan_type,,
Basic,499,799
Pro,1299,1499
Standard,799,999


**What this shows:** churners last logged in 12 to 30 days ago and stayers 1 to 9 days ago, so the two groups do not overlap on login recency. Support tickets also separate them (3 to 5 versus 0 to 2). Every Basic customer churned and no Pro or Standard customer did. Real data is almost never this tidy, so this looks like a small practice dataset.

Spend is also almost decided by plan, so it adds little on its own and may act as a proxy for income.

## 4. The non-ML baseline

The rule I chose in the memo: **flag a customer as likely to churn if they have not logged in for 10 or more days.**

I also test a few other simple rules for comparison.

In [6]:
def confusion(y_true, y_pred):
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    return tp, fp, fn, tn

def scores(tp, fp, fn, tn):
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) else float("nan"))
    return precision, recall, f1

y = df.churned

rules = {
    "Last login >= 10 days (chosen baseline)": df.last_login_days >= 10,
    "Support tickets >= 3": df.support_tickets >= 3,
    "Login >= 10 days OR tickets >= 3": (df.last_login_days >= 10) | (df.support_tickets >= 3),
    "Plan is Basic": df.plan_type == "Basic",
    "Tenure <= 8 months": df.tenure_months <= 8,
}

rows = []
for name, rule in rules.items():
    tp, fp, fn, tn = confusion(y, rule.astype(int))
    p, r, f1 = scores(tp, fp, fn, tn)
    rows.append({"rule": name, "caught (TP)": tp, "missed (FN)": fn,
                 "wrongly flagged (FP)": fp, "correct stayers (TN)": tn,
                 "precision": round(p, 2), "recall": round(r, 2), "F1": round(f1, 2)})

pd.DataFrame(rows).set_index("rule")

,caught (TP),missed (FN),wrongly flagged (FP),correct stayers (TN),precision,recall,F1
rule,,,,,,,
Last login >= 10 days (chosen baseline),5,0,0,7,1.00,1.0,1.00
Support tickets >= 3,5,0,0,7,1.00,1.0,1.00
Login >= 10 days OR tickets >= 3,5,0,0,7,1.00,1.0,1.00
Plan is Basic,5,0,0,7,1.00,1.0,1.00
Tenure <= 8 months,5,0,1,6,0.83,1.0,0.91


**What this shows:** the chosen baseline catches all 5 churners with no false alarms. Several other single rules do the same. The tenure rule is the only one that makes a mistake (it flags one customer who stayed). With a baseline this strong on this data, a model has nothing to improve on.

## 5. A small model, for demonstration only

To show how the comparison would work, I train a logistic regression on three behaviour features: `tenure_months`, `support_tickets` and `last_login_days`.

I leave out:
- `customer_id`, because it is just a label
- `monthly_spend_inr` and `plan_type`, because they are sensitive proxies for income and affordability

With 12 rows I cannot hold back a proper test set, so I use **leave-one-out cross-validation**: each customer is predicted by a model trained on the other 11. This is the fairest option available, but it is still very unstable.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, cross_val_predict

features = ["tenure_months", "support_tickets", "last_login_days"]
X = df[features]

model = make_pipeline(StandardScaler(), LogisticRegression())
loo = LeaveOneOut()

proba = cross_val_predict(model, X, y, cv=loo, method="predict_proba")[:, 1]
pred_model = (proba >= 0.5).astype(int)

tp, fp, fn, tn = confusion(y, pd.Series(pred_model, index=y.index))
p, r, f1 = scores(tp, fp, fn, tn)
print("Logistic regression (leave-one-out)")
print(f"  caught {tp}, missed {fn}, wrongly flagged {fp}, correct stayers {tn}")
print(f"  precision {p:.2f}   recall {r:.2f}   F1 {f1:.2f}")

results = df[["customer_id", "plan_type", "churned"]].copy()
results["predicted_risk"] = proba.round(3)
results["flagged"] = pred_model
results

Logistic regression (leave-one-out)
  caught 5, missed 0, wrongly flagged 0, correct stayers 7
  precision 1.00   recall 1.00   F1 1.00


,customer_id,plan_type,churned,predicted_risk,flagged
0,C001,Basic,1,0.909,1
1,C002,Pro,0,0.074,0
2,C003,Basic,1,0.599,1
3,C004,Pro,0,0.009,0
4,C005,Standard,0,0.283,0
5,C006,Basic,1,0.993,1
6,C007,Standard,0,0.050,0
7,C008,Basic,1,0.772,1
8,C009,Pro,0,0.202,0
9,C010,Basic,1,0.576,1


**What this shows:** the model also gets every customer right, matching the simple rule. It does not beat the baseline, and with 12 rows I would not trust either result for new customers. Because the groups do not overlap, almost any method would score perfectly here, which says more about the data than about the method.

**How close the calls were:** three customers sit near the 0.5 cut-off (C003 at 0.60, C010 at 0.58 and C012 at 0.49). A small change in the data could tip any of them the other way, so the perfect score is fragile. The simple login rule has no such borderline cases on this file.

**Calibration:** I am not reporting a calibration check, because 12 rows cannot support one.

## 6. Fairness check by plan type

Compare how each approach does for each plan. In this data, every Basic customer churned and no Pro or Standard customer did, so a fairness check is only partly informative, but it shows the habit I would use on real data.

In [8]:
check = df[["customer_id", "plan_type", "churned"]].copy()
check["baseline_flag"] = rules["Last login >= 10 days (chosen baseline)"].astype(int)
check["model_flag"] = pred_model

def by_group(flag_col):
    out = []
    for plan, g in check.groupby("plan_type"):
        tp, fp, fn, tn = confusion(g.churned, g[flag_col])
        out.append({"plan_type": plan, "customers": len(g),
                    "churned": int(g.churned.sum()),
                    "flagged": int(g[flag_col].sum()),
                    "missed churners (FN)": fn, "wrong flags (FP)": fp})
    return pd.DataFrame(out).set_index("plan_type")

print("Baseline rule, by plan:")
display(by_group("baseline_flag"))
print("Demonstration model, by plan:")
display(by_group("model_flag"))

Baseline rule, by plan:


,customers,churned,flagged,missed churners (FN),wrong flags (FP)
plan_type,,,,,
Basic,5,5,5,0,0
Pro,3,0,0,0,0
Standard,4,0,0,0,0


Demonstration model, by plan:


,customers,churned,flagged,missed churners (FN),wrong flags (FP)
plan_type,,,,,
Basic,5,5,5,0,0
Pro,3,0,0,0,0
Standard,4,0,0,0,0


**What this shows:** there are no mistakes in any plan group, so there is no gap to report. This is not evidence of fairness. With 3 to 5 customers per group, a gap could easily be hidden. On real data I would check this every month.

## 7. Cost of mistakes

The costs below are **my assumptions** from the memo and should be replaced with real business figures:

- **False positive** (we contact someone who would have stayed): about ₹200, roughly one month's discount plus staff time (the memo range was ₹100 to ₹300).
- **False negative** (we miss someone who then leaves): their monthly spend over the months they would have stayed. I assume 6 months.

In [9]:
FP_COST = 200          # assumption, in rupees
MONTHS_LOST = 6        # assumption: months of revenue lost per missed churner

def total_cost(flag):
    flag = pd.Series(flag, index=df.index)
    fp_mask = (flag == 1) & (y == 0)
    fn_mask = (flag == 0) & (y == 1)
    fp_cost = fp_mask.sum() * FP_COST
    fn_cost = (df.loc[fn_mask, "monthly_spend_inr"] * MONTHS_LOST).sum()
    return int(fp_mask.sum()), int(fn_mask.sum()), int(fp_cost), int(fn_cost)

# A "do nothing" option, so the numbers have something to be compared against
cost_rows = []
options = {
    "Do nothing (flag no one)": np.zeros(len(df), dtype=int),
    "Baseline: login >= 10 days": rules["Last login >= 10 days (chosen baseline)"].astype(int).values,
    "Demonstration model": pred_model,
}
for name, flag in options.items():
    fp_n, fn_n, fp_c, fn_c = total_cost(flag)
    cost_rows.append({"option": name, "wrong flags": fp_n, "missed churners": fn_n,
                      "cost of wrong flags (INR)": fp_c,
                      "cost of missed churners (INR)": fn_c,
                      "total cost (INR)": fp_c + fn_c})

pd.DataFrame(cost_rows).set_index("option")

,wrong flags,missed churners,cost of wrong flags (INR),cost of missed churners (INR),total cost (INR)
option,,,,,
Do nothing (flag no one),0,5,0,18570,18570
Baseline: login >= 10 days,0,0,0,0,0
Demonstration model,0,0,0,0,0


**What this shows:** doing nothing would cost the most, because all 5 churners would be lost. The baseline rule and the model both cost nothing on this file, since neither makes a mistake. These figures depend entirely on my assumed costs and on a perfect result from 12 rows, so I would not use them to plan a real retention budget.

## 8. Conclusion

- The data is clean, but it has only 12 customers, so it cannot support a fair model evaluation.
- A one-line rule (last login 10 or more days ago) already catches every churner with no false alarms, so a model has nothing to improve on.
- The demonstration model matches the rule but does not beat it, and both results would be unreliable on new customers.
- **Recommendation:** do not train a production ML model on this dataset. Use the simple rule, and collect a much larger dataset, with dates on every feature, before reconsidering ML.
- Before any real use, the data owner needs to confirm when each feature was measured (to rule out leakage) and whether consent exists for this use.